# 🧠 CIRCUIT 2.0 — Custom AI Model Fine-Tuning Pipeline
### Ultra Memory-Optimized QLoRA Fine-Tuning on Custom Datasets (Free Colab T4 GPU)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ei-sanu/circuit2.0/blob/main/notebooks/CIRCUIT_2_0_Fine_Tuning_Colab.ipynb)

This Google Colab notebook trains a custom **Meta Llama-3.1-8B-Instruct** model on your proprietary CIRCUIT 2.0 forensic dataset.
It automatically synchronizes with the latest datasets pushed to your GitHub repository (`https://github.com/ei-sanu/circuit2.0`).

**Trained Intelligence Capabilities:**
1. **Bilingual FIR Extraction & NER** (IPC, BNS 318/61, PMLA, suspect PAN/vehicles)
2. **Structuring & Smurfing Anomaly Detection** (Sub-₹50k PAN evasion under Income Tax Act Sec 139A)
3. **Call Detail Records (CDR) Nocturnal Surveillance** (Off-hours cell tower handshakes)
4. **Syndicate Bottleneck Disruption** (Network articulation points & PageRank)
5. **Section 63 BSA Evidence Certification** (Cryptographic SHA-256 electronic evidence certificates)
6. **Circular Hawala Loops & Round-Tripping** (Closed-loop multi-hop laundering chains)

### Step 1: Select GPU Runtime & Install Dependencies
In Colab menu: Click **Runtime** ➔ **Change runtime type** ➔ Select **T4 GPU** (Free) or **A100 GPU**.

In [ ]:
!pip install -q --upgrade pip
!pip install -q torch transformers peft trl bitsandbytes datasets accelerate
import torch, gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print(f"[+] GPU Available: {torch.cuda.is_available()} - Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

### Step 2: Run Ultra-Low Memory QLoRA Fine-Tuning (4-Bit NF4 + Automatic GitHub Sync / In-Memory Dataset)

In [ ]:
import os, gc, random, json, urllib.request, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import Dataset, load_dataset
from trl import SFTTrainer

# 1. Clean CUDA memory cache
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

BASE_MODEL = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"
OUTPUT_DIR = "checkpoints/circuit_llama3_lora"

print(f"[*] Loading tokenizer: {BASE_MODEL}...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 2. Strict 4-Bit NF4 Quantization (Only ~4.2 GB VRAM on T4)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f"[*] Loading model in 4-bit NF4: {BASE_MODEL}...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

# 3. LoRA Configuration
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)
model = get_peft_model(model, peft_config)

# 4. Hybrid Dataset Loader: Auto-sync from GitHub or generate dynamically
def generate_large_forensic_dataset(count=1000):
    indian_cities = [
        "Mumbai", "Delhi", "Bengaluru", "Kolkata", "Ahmedabad", "Pune", "Hyderabad",
        "Chennai", "Jaipur", "Lucknow", "Chandigarh", "Surat", "Indore", "Kochi"
    ]
    suspect_names = [
        "Vikram Rathore", "Karan Singhania", "Sunil Gounder", "Rashid Al-Mansoor",
        "Rajesh Verma", "Amit Deshmukh", "Sameer Khan", "Harpreet Gill", "Dinesh Kulkarni",
        "Manoj Singhal", "Tariq Al-Mansoor", "Anand Singhal"
    ]
    aliases = ["RV", "Cargo Raja", "Bhaiya", "Doctor", "Hawala King", "Captain", "Cash Baba", "Shadow", "Falcon"]
    statutes_pool = ["IPC 420", "IPC 120B", "IPC 468", "IPC 471", "BNS 318", "BNS 61", "PMLA Sec 3", "NDPS Sec 21", "Arms Act 25", "IT Act 66D"]

    samples = [
        {
            "instruction": "Analyze the financial transaction chain and identify any anti-money laundering anomaly signatures.",
            "input": "Account: ACC-918237 (Rajesh Verma)\nTransactions: Received ₹48,500 from ACC-102938 and ₹49,000 from ACC-771239 within 4 hours; immediately transferred ₹95,000 to ACC-441029 (Apex Factoring LLC).\nDispersal Window: < 24 hours.",
            "output": "CRITICAL ANOMALY DETECTED: Structuring / Smurfing with Pass-Through Mule Dispersal.\n\n1. Structuring (PAN Evasion): Repetitive transfers placed just under the ₹50,000 statutory reporting threshold under Section 139A of the Income Tax Act.\n2. Pass-Through Routing: 97.4% fund dispersal within 4 hours indicates layering through a conduit mule account.\n3. Recommendation: Freeze beneficiary account under PMLA Section 17 and request KYC audit on originating accounts."
        },
        {
            "instruction": "Perform bilingual Named Entity Recognition (NER) and statutory extraction on the First Information Report (FIR).",
            "input": "FIR Document Text:\n\"\"\"FIR No. 124/2026 registered at Colaba Cyber Crime PS, Mumbai. Accused Rajesh Verma alias 'RV' (PAN: ABCPV9182K, Mobile: +91 98201 44891) coordinated Hawala cash handshakes u/s IPC 420, 120B, and Section 318 BNS.\"\"\"",
            "output": "{\n  \"fir_number\": \"124/2026\",\n  \"police_station\": \"Colaba Cyber Crime PS, Mumbai\",\n  \"suspects\": [{\"name\": \"Rajesh Verma\", \"aliases\": [\"RV\"], \"pan\": \"ABCPV9182K\", \"phone\": \"+919820144891\"}],\n  \"statutes\": [\"IPC 420\", \"IPC 120B\", \"BNS 318\"],\n  \"offense_type\": \"Hawala Diversion & Cheating\"\n}"
        },
        {
            "instruction": "Identify the primary bottleneck node in the following syndicate graph and suggest a disruption strategy.",
            "input": "Syndicate: Operation Nightfall (CR-2026-00124)\nArticulation Points: [P-0001: Rajesh Verma]\nBetweenness Centrality: Rajesh Verma (0.78), Tariq Al-Mansoor (0.64)\nConnected Clusters: Mumbai Logistics <---> Dubai Bullion Vaults.",
            "output": "SYNDICATE BOTTLENECK DISRUPTION ANALYSIS:\n• Key Bottleneck: Rajesh Verma (P-0001).\n• Graph Topology: Sole articulation bridge connecting domestic forwarders with international clearing houses.\n• Disruption Impact: Neutralizing Node P-0001 increases network fragmentation by 86%, immediately isolating domestic cells from offshore liquidity.\n• Strategy: Execute simultaneous freeze on P-0001 banking conduits before apprehending."
        },
        {
            "instruction": "Generate a Section 63 BSA / Section 65B Indian Evidence Act certification summary for digital transaction evidence.",
            "input": "Case: CR-2026-00124 (Operation Nightfall)\nSignatory Officer: Inspector Vikram Rao (Badge: MH-CID-8821)\nEvidence Ingested: 14 Bank CSV Ledgers, 8 Scanned FIRs, 42 Telecom CDR Records",
            "output": "STATUTORY ELECTRONIC EVIDENCE CERTIFICATION:\n\n\"I, Inspector Vikram Rao (Badge: MH-CID-8821), Investigating Officer, State Cyber Crime Unit, hereby certify under Section 63 of Bharatiya Sakshya Adhiniyam, 2023 (formerly Section 65B of Indian Evidence Act, 1872) that the electronic records, relationship graphs, and financial ledgers for Case CR-2026-00124 were extracted using the CIRCUIT 2.0 Intelligence Platform during ordinary operational parameters. SHA-256 Digest is authentic and unaltered.\""
        }
    ]

    for i in range(count):
        s_name = random.choice(suspect_names)
        alias = random.choice(aliases)
        city = random.choice(indian_cities)
        amt = random.randint(20, 95) * 1000
        total_cr = round(random.uniform(0.5, 9.8), 2)
        station = f"{city} Cyber Crime Police Station"
        statutes = random.sample(statutes_pool, k=random.randint(2, 4))

        mode = i % 5
        if mode == 0:
            samples.append({
                "instruction": "Analyze the financial transaction chain and identify any anti-money laundering anomaly signatures.",
                "input": f"Account: ACC-{random.randint(100000, 999999)} ({s_name})\nTransactions: Repetitive deposits of ₹{amt:,} and ₹{amt - 1200:,} aggregating to ₹{total_cr} Crore received from 4 distinct accounts in {city} within 8 hours.",
                "output": f"CRITICAL ANOMALY DETECTED: Structuring / Smurfing Pattern.\n\n1. Sub-Threshold Evasion: Transfers placed below statutory ₹50,000 PAN threshold to evade automated CTR filing.\n2. Aggregate Turnover: ₹{total_cr} Crore laundered through distributed micro-deposits in {city}.\n3. Relevant Law: PMLA Section 3 and Income Tax Act Section 139A.\n4. Action: Initiate immediate SAR filing and freeze conduit accounts."
            })
        elif mode == 1:
            samples.append({
                "instruction": "Perform bilingual Named Entity Recognition (NER) and statutory extraction on the First Information Report (FIR).",
                "input": f"FIR No. {random.randint(10, 500)}/2026 filed at {station}. Accused {s_name} (Alias: '{alias}') booked for commercial fraud under {', '.join(statutes)}.",
                "output": f"FIR EXTRACTION REPORT:\n• Police Station: {station}\n• Accused: {s_name} (Alias: '{alias}')\n• Statutory Sections: {', '.join(statutes)}\n• Total Value: ₹{total_cr} Crore\n• Jurisdiction: {city}"
            })
        elif mode == 2:
            samples.append({
                "instruction": "Evaluate the Call Detail Records (CDR) and identify covert telecommunication patterns.",
                "input": f"Target: {s_name}\nLocation: {city} Tower {random.randint(100, 999)}\nCall Timings: 01:45 AM, 03:10 AM, 04:30 AM (Duration: {random.randint(300, 1500)}s).",
                "output": f"TELECOM SURVEILLANCE REPORT:\n• Anomaly: Nocturnal Intercept Spike ({city} Sector).\n• Intelligence Analysis: Concentrated nocturnal communication indicates active logistical coordination.\n• Action: Cross-reference cell tower dump and issue Section 91 CrPC notice."
            })
        elif mode == 3:
            samples.append({
                "instruction": "Explain the role and structural influence of the entity within the criminal syndicate network.",
                "input": f"Entity: {s_name} (Alias: {alias})\nCase: CR-2026-{random.randint(100, 999):05d} ({city} Sector)\nConnected Nodes: 6 Mule Accounts, 2 Transit Warehouses.",
                "output": f"SYNDICATE EGO-NETWORK DOSSIER:\n• Entity: {s_name} ('{alias}')\n• Operational Role: Regional Transport & Factoring Coordinator in {city}.\n• Graph Position: High betweenness broker connecting ground mules to master account holders.\n• Prosecution Strategy: Freeze linked accounts under PMLA Section 17 prior to apprehension."
            })
        else:
            samples.append({
                "instruction": "Analyze a Circular Hawala Loop detected across bank accounts and explain the flow topology.",
                "input": f"Transactions: ACC-A ({s_name}) -> ACC-B ({city} Bullion) -> ACC-C (Dubai Logistics) -> ACC-A (₹{total_cr} Crore total turnover).",
                "output": f"CIRCULAR HAWALA FLOW FORENSIC SUMMARY:\n• Typology: Closed-Loop Round-Tripping ({city} <-> Dubai).\n• Volume: ₹{total_cr} Crore.\n• Purpose: Simulating legitimate commercial activity to repatriate untaxed illicit capital back to the originating entity.\n• Evidence Flag: High risk under PMLA Section 3."
            })
    return samples

def format_sample(s):
    return (
        "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n"
        "You are CIRCUIT AI, a specialized criminal intelligence and financial forensic analysis copilot.<|eot_id|>\n"
        "<|start_header_id|>user<|end_header_id|>\n"
        f"{s['instruction']}\n\nContext:\n{s.get('input', '')}<|eot_id|>\n"
        "<|start_header_id|>assistant<|end_header_id|>\n"
        f"{s['output']}<|eot_id|>"
    )

# Attempt loading dataset from GitHub repo; otherwise use dynamic generator
train_file = "data/train.jsonl"
val_file = "data/val.jsonl"
os.makedirs("data", exist_ok=True)

if not os.path.exists(train_file):
    try:
        print("[*] Checking GitHub for latest pushed dataset...")
        urllib.request.urlretrieve("https://raw.githubusercontent.com/ei-sanu/circuit2.0/main/data/train.jsonl", train_file)
        urllib.request.urlretrieve("https://raw.githubusercontent.com/ei-sanu/circuit2.0/main/data/val.jsonl", val_file)
    except Exception as e:
        print(f"[!] GitHub download fallback: {e}")

if os.path.exists(train_file) and os.path.getsize(train_file) > 100:
    print(f"[+] Ingested latest custom dataset from GitHub: {train_file}")
    raw_train = [json.loads(l) for l in open(train_file, encoding="utf-8") if l.strip()]
    raw_val = [json.loads(l) for l in open(val_file, encoding="utf-8") if l.strip()]
    train_ds = Dataset.from_list([{"text": format_sample(s)} for s in raw_train])
    val_ds = Dataset.from_list([{"text": format_sample(s)} for s in raw_val])
else:
    print("[*] Generating 1,000+ domain-diversified forensic intelligence pairs in memory...")
    all_samples = generate_large_forensic_dataset(count=1000)
    random.shuffle(all_samples)
    split_idx = int(len(all_samples) * 0.85)
    train_ds = Dataset.from_list([{"text": format_sample(s)} for s in all_samples[:split_idx]])
    val_ds = Dataset.from_list([{"text": format_sample(s)} for s in all_samples[split_idx:]])

print(f"[+] Loaded {len(train_ds)} Training and {len(val_ds)} Validation pairs in memory!")

# 5. Optimized Training Settings (Batch=1, Accum=8 for T4 GPU)
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    learning_rate=2e-4,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    fp16=True,
    optim="paged_adamw_8bit",
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    dataset_text_field="text",
    max_seq_length=1024,
    tokenizer=tokenizer,
    args=training_args,
    peft_config=peft_config,
)

print("[*] Starting QLoRA Fine-Tuning on GPU (VRAM ~4.2 GB)...")
trainer.train()
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"\n[+] Training complete! Fine-tuned adapters saved to {OUTPUT_DIR}")

### Step 3: Test Real-Time Interactive Inference (ChatGPT-Style)
Ask your fine-tuned model any investigative or natural language query:

In [ ]:
def ask_circuit_model(user_query, context=""):
    prompt = (
        "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n"
        "You are CIRCUIT AI, an expert criminal intelligence copilot.<|eot_id|>\n"
        "<|start_header_id|>user<|end_header_id|>\n"
        f"{user_query}\n\nContext:\n{context}<|eot_id|>\n"
        "<|start_header_id|>assistant<|end_header_id|>\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=350, temperature=0.2, top_p=0.9)
    return tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

# Test Query 1: Structuring & Smurfing
print("--- TEST 1: SMURFING DETECTION ---")
print(ask_circuit_model(
    user_query="Analyze the financial transaction chain and check for anti-money laundering anomalies.",
    context="Account: ACC-918237 (Rajesh Verma)\nDeposits: Received ₹48,500 and ₹49,000 within 3 hours from 2 distinct accounts; immediately routed ₹95,000 to Apex Factoring LLC."
))

# Test Query 2: Syndicate Bottleneck
print("\n--- TEST 2: BOTTLENECK DISRUPTION ---")
print(ask_circuit_model(
    user_query="Identify the primary bottleneck node and suggest a disruption strategy.",
    context="Syndicate: Operation Nightfall\nArticulation Points: [P-0001: Rajesh Verma]\nBetweenness Centrality: 0.78 (Connecting Mumbai Port Logistics to Dubai Vaults)"
))

# Test Query 3: Statutory Evidence Certification
print("\n--- TEST 3: EVIDENCE CERTIFICATION ---")
print(ask_circuit_model(
    user_query="Generate a Section 63 BSA / Section 65B Indian Evidence Act certification summary.",
    context="Case: CR-2026-00124 (Operation Nightfall)\nSignatory: Inspector Vikram Rao (MH-CID-8821)"
))

### Step 4: Download Trained Model Weights to Your Local Machine
Run this cell to zip and download your trained adapter weights to drop into CIRCUIT 2.0.

In [ ]:
!zip -r circuit_model_lora.zip checkpoints/circuit_llama3_lora
from google.colab import files
files.download('circuit_model_lora.zip')
print("[+] Download initiated! Extract circuit_model_lora.zip into your local CIRCUIT-2.0/checkpoints/ directory.")